####Requirement
Read data from flight_time and transform as below
1. Rename fl_date to dep_date
2. Compute arr_date
3. Following fields to represent full timestamp
    1. crs_dep_time
    2. dep_time
    3. crs_arr_time
    4. arr_time



In [0]:
%sql

select * from dev.spark_db.flight_time order by DISTANCE desc limit 10

FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
2000-01-03,CO,14,HNL,"Honolulu, HI",EWR,"Newark, NJ",INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '10:28' HOUR TO MINUTE,INTERVAL '05' MINUTE,INTERVAL '10:20' HOUR TO MINUTE,INTERVAL '10:33' HOUR TO MINUTE,0,4962
2000-01-04,CO,14,HNL,"Honolulu, HI",EWR,"Newark, NJ",INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '20:19' HOUR TO MINUTE,INTERVAL '10:21' HOUR TO MINUTE,INTERVAL '08' MINUTE,INTERVAL '10:20' HOUR TO MINUTE,INTERVAL '10:29' HOUR TO MINUTE,0,4962
2000-01-03,CO,15,EWR,"Newark, NJ",HNL,"Honolulu, HI",INTERVAL '08:05' HOUR TO MINUTE,INTERVAL '08:35' HOUR TO MINUTE,INTERVAL '13:46' HOUR TO MINUTE,INTERVAL '04' MINUTE,INTERVAL '13:40' HOUR TO MINUTE,INTERVAL '13:50' HOUR TO MINUTE,0,4962
2000-01-05,CO,15,EWR,"Newark, NJ",HNL,"Honolulu, HI",INTERVAL '08:05' HOUR TO MINUTE,INTERVAL '08:01' HOUR TO MINUTE,INTERVAL '13:30' HOUR TO MINUTE,INTERVAL '04' MINUTE,INTERVAL '13:40' HOUR TO MINUTE,INTERVAL '13:34' HOUR TO MINUTE,0,4962
2000-01-01,CO,14,HNL,"Honolulu, HI",EWR,"Newark, NJ",INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '20:14' HOUR TO MINUTE,INTERVAL '10:14' HOUR TO MINUTE,INTERVAL '12' MINUTE,INTERVAL '10:20' HOUR TO MINUTE,INTERVAL '10:26' HOUR TO MINUTE,0,4962
2000-01-02,CO,15,EWR,"Newark, NJ",HNL,"Honolulu, HI",INTERVAL '08:05' HOUR TO MINUTE,INTERVAL '09:25' HOUR TO MINUTE,INTERVAL '14:49' HOUR TO MINUTE,INTERVAL '03' MINUTE,INTERVAL '13:40' HOUR TO MINUTE,INTERVAL '14:52' HOUR TO MINUTE,0,4962
2000-01-01,CO,15,EWR,"Newark, NJ",HNL,"Honolulu, HI",INTERVAL '08:05' HOUR TO MINUTE,INTERVAL '08:05' HOUR TO MINUTE,INTERVAL '13:35' HOUR TO MINUTE,INTERVAL '04' MINUTE,INTERVAL '13:40' HOUR TO MINUTE,INTERVAL '13:39' HOUR TO MINUTE,0,4962
2000-01-02,CO,14,HNL,"Honolulu, HI",EWR,"Newark, NJ",INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '20:11' HOUR TO MINUTE,INTERVAL '10:25' HOUR TO MINUTE,INTERVAL '08' MINUTE,INTERVAL '10:20' HOUR TO MINUTE,INTERVAL '10:33' HOUR TO MINUTE,0,4962
2000-01-04,CO,15,EWR,"Newark, NJ",HNL,"Honolulu, HI",INTERVAL '08:05' HOUR TO MINUTE,INTERVAL '08:04' HOUR TO MINUTE,INTERVAL '13:20' HOUR TO MINUTE,INTERVAL '04' MINUTE,INTERVAL '13:40' HOUR TO MINUTE,INTERVAL '13:24' HOUR TO MINUTE,0,4962
2000-01-05,CO,14,HNL,"Honolulu, HI",EWR,"Newark, NJ",INTERVAL '20:15' HOUR TO MINUTE,INTERVAL '20:19' HOUR TO MINUTE,INTERVAL '10:35' HOUR TO MINUTE,INTERVAL '08' MINUTE,INTERVAL '10:20' HOUR TO MINUTE,INTERVAL '10:43' HOUR TO MINUTE,0,4962


1. Can we do it using select() or selectExpr() transformations?

In [0]:
flight_time_df = spark.read.table("dev.spark_db.flight_time")

flight_time_1_df = (
    flight_time_df.selectExpr(
        "fl_date as dep_date",
        "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
        "dep_date + crs_dep_time as crs_dep_time",
        "dep_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time",
        "op_carrier"
    )
)

flight_time_1_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

dep_date,arr_date,crs_dep_time,dep_time,crs_arr_time,arr_time,op_carrier
2000-01-01,2000-01-02,2000-01-01T11:15:00.000Z,2000-01-01T11:13:00.000Z,2000-01-02T14:00:00.000Z,2000-01-02T13:48:00.000Z,DL
2000-01-01,2000-01-02,2000-01-01T18:50:00.000Z,2000-01-01T19:42:00.000Z,2000-01-02T21:55:00.000Z,2000-01-02T22:33:00.000Z,AA
2000-01-01,2000-01-01,2000-01-01T09:40:00.000Z,2000-01-01T09:40:00.000Z,2000-01-01T10:35:00.000Z,2000-01-01T10:37:00.000Z,WN


2. Can we do it using withColumn() or withColumns()?

In [0]:
from pyspark.sql.functions import expr

flight_time_2_df = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", expr("to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date"))
      .withColumns({
        "crs_dep_time": expr("dep_date + crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": expr("arr_date + arr_time"),
      })
)

flight_time_2_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

dep_date,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,crs_dep_time,dep_time,WHEELS_ON,TAXI_IN,crs_arr_time,arr_time,CANCELLED,DISTANCE,arr_date
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",2000-01-01T11:15:00.000Z,2000-01-01T11:13:00.000Z,INTERVAL '13:43' HOUR TO MINUTE,INTERVAL '05' MINUTE,2000-01-02T14:00:00.000Z,2000-01-02T13:48:00.000Z,0,946,2000-01-02
2000-01-01,AA,1451,ORD,"Chicago, IL",ATL,"Atlanta, GA",2000-01-01T18:50:00.000Z,2000-01-01T19:42:00.000Z,INTERVAL '22:24' HOUR TO MINUTE,INTERVAL '09' MINUTE,2000-01-02T21:55:00.000Z,2000-01-02T22:33:00.000Z,0,606,2000-01-02
2000-01-01,WN,1451,LAS,"Las Vegas, NV",BUR,"Burbank, CA",2000-01-01T09:40:00.000Z,2000-01-01T09:40:00.000Z,INTERVAL '10:36' HOUR TO MINUTE,INTERVAL '01' MINUTE,2000-01-01T10:35:00.000Z,2000-01-01T10:37:00.000Z,0,223,2000-01-01


3. Alternative approach to write expressions\ - like SQL function
  Why to use it?
    * It gives access to column functions

In [0]:
from pyspark.sql.functions import to_date, col

flight_time_2_df = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", to_date(col("dep_date") + col("dep_time") + col("wheels_on") + col("taxi_in")))
      .withColumns({
        "crs_dep_time": col("dep_date") + col("crs_dep_time"),
        "dep_time": col("dep_date") + col("dep_time"),
        "crs_arr_time": col("arr_date") + col("crs_arr_time"),
        "arr_time": col("arr_date") + col("arr_time"),
      })
)

flight_time_2_df.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).display()

dep_date,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,crs_dep_time,dep_time,WHEELS_ON,TAXI_IN,crs_arr_time,arr_time,CANCELLED,DISTANCE,arr_date
2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",2000-01-01T11:15:00.000Z,2000-01-01T11:13:00.000Z,INTERVAL '13:43' HOUR TO MINUTE,INTERVAL '05' MINUTE,2000-01-02T14:00:00.000Z,2000-01-02T13:48:00.000Z,0,946,2000-01-02
2000-01-01,AA,1451,ORD,"Chicago, IL",ATL,"Atlanta, GA",2000-01-01T18:50:00.000Z,2000-01-01T19:42:00.000Z,INTERVAL '22:24' HOUR TO MINUTE,INTERVAL '09' MINUTE,2000-01-02T21:55:00.000Z,2000-01-02T22:33:00.000Z,0,606,2000-01-02
2000-01-01,WN,1451,LAS,"Las Vegas, NV",BUR,"Burbank, CA",2000-01-01T09:40:00.000Z,2000-01-01T09:40:00.000Z,INTERVAL '10:36' HOUR TO MINUTE,INTERVAL '01' MINUTE,2000-01-01T10:35:00.000Z,2000-01-01T10:37:00.000Z,0,223,2000-01-01
